<a href="https://colab.research.google.com/github/toanpt74/Access-Control-Allow-Origin/blob/master/VGG16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import glob
import os, sys
import random
from tqdm import tqdm

import numpy as np

from keras import layers
from keras.models import Sequential
from keras.layers import Dropout, Flatten, Dense, ZeroPadding2D
from keras import applications
import pandas as pd
import matplotlib.pyplot as plt
from keras import backend as K
import cv2
import tensorflow as tf
from sklearn.model_selection import train_test_split
from keras import models
from keras.utils import load_img, img_to_array

train_data_dir = r'D:\Non_Documents\ToanPT\Program\KnapSackProblem\data\cat-dog\working\train'
test_data_dir = r'D:\Non_Documents\ToanPT\Program\KnapSackProblem\data\cat-dog\working\test'
vgg_model_path = 'model/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5'
epochs = 20
batch_size = 20
img_width, img_height = 150, 150
training_n_bound = 5000  # set to None to use the entire training dataset; it took about 2 hours at my Macbook Pro.
NB_CLASSES=2

def VGG16():
    model = models.Sequential()
    model.add(layers.ZeroPadding2D((1, 1), input_shape=(224, 224, 3)))
    model.add(layers.Convolution2D(64, (3, 3), activation='relu'))
    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(64, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2), strides=(2, 2)))

    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(128, (3, 3), activation='relu'))
    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(128, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2), strides=(2, 2)))

    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(256, (3, 3), activation='relu'))
    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(256, (3, 3), activation='relu'))
    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(256, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2), strides=(2, 2)))

    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(512, (3, 3), activation='relu'))
    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(512, (3, 3), activation='relu'))
    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(512, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2), strides=(2, 2)))

    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(512, (3, 3), activation='relu'))
    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(512, (3, 3), activation='relu'))
    model.add(layers.ZeroPadding2D((1, 1)))
    model.add(layers.Convolution2D(512, (3, 3), activation='relu'))
    model.add(layers.MaxPooling2D((2, 2), strides=(2, 2)))

    model.add(layers.Flatten())
    # top layer of the VGG net
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(128, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(NB_CLASSES, activation='softmax'))


def gen_image_label(directory):
    ''' A generator that tuple (label, id, jpg_filename).'''
    for root, dirs, files in os.walk(directory):
        for f in files:
            _, ext = os.path.splitext(f)
            if ext != '.jpg':
                continue
            basename = os.path.basename(f)
            splits = basename.split('.')
            if len(splits) == 3:
                label, id_, ext = splits
            else:
                label = None
                id_, ext = splits
            fullname = os.path.join(root, f)
            yield label, int(id_), fullname

lst = list(gen_image_label(train_data_dir))
random.shuffle(lst)

training_n_bound = len(lst)
if training_n_bound is not None:
    lst = lst[:training_n_bound]

train_df = pd.DataFrame(lst, columns=['label', 'id', 'filename'])
#train_df = train_df.sort_values(by=['label', 'id'])
train_df['label_code'] = train_df.label.map({'cat':0, 'dog':1})

lst_test = list(gen_image_label(test_data_dir))
training_n_bound = len(lst_test)
if training_n_bound is not None:
    lst_test = lst_test[:training_n_bound]
test_df = pd.DataFrame(lst_test, columns=['label', 'id', 'filename'])
#test_df = test_df.sort_values(by=['label', 'id'])
test_df['label_code'] = test_df.label.map({'cat':0, 'dog':1})

def gen_label_image_batch(df):
    img_arrays = []
    label_arrays = []
    target_label =[]
    for index, row in df.iterrows():
        #img =load_img(row['filename'],target_size=(img_width, img_height))
        img = cv2.resize(cv2.imread(row['filename'], cv2.IMREAD_COLOR), (img_width, img_height))
        img = np.array(img)
        img = img.astype('float32')/255
        img_arrays.append(img)
        label_arrays.append(row['label_code'])
        target_label.append(row['label'])
    img_arrays = np.array(img_arrays)
    label_arrays = np.array(label_arrays)
    target_label = np.array(target_label)
    return img_arrays, label_arrays, target_label

def display_images(label, n=5):
    fig = plt.figure(figsize=(16, 8))
    for j, fn in enumerate(train_df.loc[train_df.label == label].head(n).filename):
        img = load_img(fn, target_size=(img_width, img_height))
        fig.add_subplot(1, n, j + 1)
        f = plt.imshow(img)
        f.axes.get_xaxis().set_visible(False)
        f.axes.get_yaxis().set_visible(False)
        plt.title(label)
    plt.show()

X_data, y_data, train_label = gen_label_image_batch(train_df)
X_train, X_test, y_train_label, y_test_label = train_test_split(X_data, y_data, shuffle=True, test_size=0.2, random_state=42)
y_train_label = tf.keras.utils.to_categorical(y_train_label, NB_CLASSES)
y_test_label = tf.keras.utils.to_categorical(y_test_label, NB_CLASSES)

plt.figure(figsize=(20,9))
pos=0
for i in range(5):
    pos=pos+1
    plt.subplot(4,5,pos)
    plt.imshow(cv2.cvtColor(X_train[i], cv2.COLOR_RGB2BGR))
    plt.title(y_train_label[i])
plt.subplots_adjust(hspace=0.4,wspace=0.4)
plt.show()







